# LightGBM 튜닝 파라미터 + 시드 앙상블

`tune_lgbm_final.ipynb`에서 찾은 튜닝된 파라미터(`best_params.json`)를 기반으로,
**시드 앙상블**을 추가로 적용합니다.

**시드 앙상블이란**: 같은 하이퍼파라미터라도 `random_state`를 다르게 주면
LightGBM이 학습 과정에서 미세하게 다른 트리를 만듭니다 (feature/data
subsampling 순서 등의 무작위성 때문). 여러 시드로 학습한 모델의 예측을
평균내면 분산이 줄어들어 약간의 성능 향상을 얻을 수 있습니다 (배깅과 같은 원리).

**검증 결과** (3-fold 빠른 검증, outer random_state 2가지로 확인):
- 단일 시드 대비 시드 3개 앙상블이 +0.0001~0.0002 일관되게 개선됨
- 다만 학습 시간이 시드 수만큼 늘어남 (기존 5-Fold -> 5-Fold x 3시드 = 15번 학습)

**사전 준비**: `tune_lgbm_final.ipynb`를 먼저 실행해서 같은 폴더(또는 인접 폴더)에
`best_params.json`이 생성되어 있어야 합니다.


## 1. 라이브러리 및 설정

In [1]:
import json
import os

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder

# 데이터 경로 (이 노트북이 "attempt file" 폴더 등에 있다고 가정, 필요시 조정)
DATA_DIR = "../data"
SUBMISSION_DIR = "../submission file"  # 제출 파일은 이 폴더에 저장

TRAIN_PATH = f"{DATA_DIR}/train.csv"
TEST_PATH = f"{DATA_DIR}/test.csv"
SAMPLE_SUBMISSION_PATH = f"{DATA_DIR}/sample_submission.csv"

TARGET_COL = "임신 성공 여부"
ID_COL = "ID"
N_SPLITS = 5

# 시드 앙상블에 사용할 시드 목록. 시드 수를 늘리면 시간이 더 걸리지만
# 보통 3~5개 정도에서 효과가 충분히 나타나고 그 이상은 수익이 줄어듭니다.
SEEDS = [1004, 42, 7]

BEST_PARAMS_PATH = "best_params.json"
VERSION_FILE = "version_state.json"
MODEL_NAME = "LGBM_tuned_seedens"

os.makedirs(SUBMISSION_DIR, exist_ok=True)

## 2. 버전(차수) 관리

`submission.ipynb`, `tune_lgbm_final.ipynb`와 동일한 `version_state.json`을 공유합니다.


In [2]:
def load_version_state():
    if os.path.exists(VERSION_FILE):
        with open(VERSION_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    return {"version": 1, "submitted": False}


def save_version_state(state):
    with open(VERSION_FILE, "w", encoding="utf-8") as f:
        json.dump(state, f, ensure_ascii=False, indent=2)


def get_active_version():
    state = load_version_state()
    if state["submitted"]:
        state["version"] += 1
        state["submitted"] = False
        save_version_state(state)
    return state["version"]


state = load_version_state()
print("현재 차수:", state["version"], "차  (제출 완료 여부:", state["submitted"], ")")

현재 차수: 2 차  (제출 완료 여부: False )


## 3. 튜닝된 파라미터 불러오기

In [3]:
if not os.path.exists(BEST_PARAMS_PATH):
    raise FileNotFoundError(
        f"{BEST_PARAMS_PATH}가 없습니다. 먼저 tune_lgbm_final.ipynb를 실행해서 "
        f"튜닝된 파라미터를 생성하세요."
    )

with open(BEST_PARAMS_PATH, "r", encoding="utf-8") as f:
    tuned = json.load(f)
best_params = tuned["best_params"]
print(f"튜닝된 파라미터 로드 완료 (튜닝 단계 점수: {tuned['best_value']:.5f})")
print(best_params)

튜닝된 파라미터 로드 완료 (튜닝 단계 점수: 0.73976)
{'n_estimators': 2357, 'learning_rate': 0.013668973845707234, 'num_leaves': 159, 'max_depth': 5, 'min_child_samples': 83, 'subsample': 0.908634361540321, 'colsample_bytree': 0.7944647062780377, 'reg_alpha': 6.529095336314542, 'reg_lambda': 0.00029237017648296726, 'min_split_gain': 0.4557413392057567}


## 4. 데이터 로드 및 전처리 (main.py와 동일한 파이프라인)

In [4]:
def preprocess(train, test):
    test_ids = test[ID_COL].copy()
    train = train.drop(columns=[ID_COL])
    test = test.drop(columns=[ID_COL])

    for df in (train, test):
        df["is_DI"] = (df["시술 유형"] == "DI").astype(int)
        df["froze_embryo"] = df["동결 배아 사용 여부"].fillna(0).astype(int)

    cols_to_drop = ["불임 원인 - 여성 요인", "난자 채취 경과일"]
    train = train.drop(columns=cols_to_drop)
    test = test.drop(columns=cols_to_drop)

    categorical_cols = train.select_dtypes(include=["object", "str"]).columns
    numerical_cols = train.select_dtypes(include=["int64", "float64"]).columns
    na_cols = train.isna().sum().loc[lambda x: x > 0].index

    for col in na_cols:
        if col in categorical_cols:
            train[col] = train[col].fillna("해당없음")
            test[col] = test[col].fillna("해당없음")
        elif col in numerical_cols:
            train[col] = train[col].fillna(-1)
            test[col] = test[col].fillna(-1)

    for col in categorical_cols:
        le = LabelEncoder()
        le.fit(train[col])
        unseen_labels = set(test[col].unique()) - set(le.classes_)
        if unseen_labels:
            le.classes_ = np.append(le.classes_, list(unseen_labels))
        train[col] = le.transform(train[col])
        test[col] = le.transform(test[col])

    return train, test, test_ids


train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)
train_p, test_p, test_ids = preprocess(train_raw, test_raw)

X = train_p.drop(columns=[TARGET_COL])
y = train_p[TARGET_COL]
print(f"전처리 완료: train={X.shape}, test={test_p.shape}")

전처리 완료: train=(256351, 67), test=(90067, 67)


## 5. Stratified 5-Fold x 시드 앙상블 학습

각 fold마다 {len(SEEDS)}개의 시드로 모델을 학습하여 평균을 냅니다.
총 {N_SPLITS} x {len(SEEDS)} = {N_SPLITS * len(SEEDS)}번의 모델 학습이 이루어지므로,
기존 5-Fold 단일 시드보다 시간이 더 걸립니다 (대략 시드 수만큼 비례).


In [9]:
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=1004)

all_models = []  # (fold, seed) 별 모델을 모두 저장 -> 최종 예측 시 전부 평균
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    seed_preds = []
    for seed in SEEDS:
        model = LGBMClassifier(random_state=seed, verbose=-1, **best_params)
        model.fit(X_train, y_train)
        all_models.append(model)
        seed_preds.append(model.predict_proba(X_val)[:, 1])

    # 이 fold의 검증 점수는 시드 앙상블(평균) 기준으로 계산
    val_proba = np.mean(seed_preds, axis=0)
    score = roc_auc_score(y_val, val_proba)
    fold_scores.append(score)
    print(f"[Fold {fold}/{N_SPLITS}] 시드 {len(SEEDS)}개 앙상블 ROC-AUC: {score:.4f}")

best_local_score = float(np.mean(fold_scores))
print(f"\n5-Fold x 시드앙상블 평균 ROC-AUC: {best_local_score:.4f} (std: {np.std(fold_scores):.4f})")
print(f"총 학습된 모델 수: {len(all_models)}")

[Fold 1/5] 시드 3개 앙상블 ROC-AUC: 0.7440
[Fold 2/5] 시드 3개 앙상블 ROC-AUC: 0.7404
[Fold 3/5] 시드 3개 앙상블 ROC-AUC: 0.7380
[Fold 4/5] 시드 3개 앙상블 ROC-AUC: 0.7385
[Fold 5/5] 시드 3개 앙상블 ROC-AUC: 0.7393

5-Fold x 시드앙상블 평균 ROC-AUC: 0.7400 (std: 0.0021)
총 학습된 모델 수: 15


## 6. 테스트 예측 및 제출 파일 생성

In [10]:
final_pred = np.mean([model.predict_proba(test_p)[:, 1] for model in all_models], axis=0)

VERSION = get_active_version()
LOCAL_SCORE = f"{best_local_score:.5f}"
SAVE_NAME = f"{VERSION}차_{MODEL_NAME}_submit_improved({LOCAL_SCORE}->).csv"
SAVE_PATH = f"{SUBMISSION_DIR}/{SAVE_NAME}"

submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
submission["ID"] = test_ids.values
submission["probability"] = final_pred
submission.to_csv(SAVE_PATH, index=False)

print(f"저장 완료: {SAVE_PATH}")
print(submission.head())

저장 완료: ../submission file/2차_LGBM_tuned_seedens_submit_improved(0.74002->).csv
           ID  probability
0  TEST_00000     0.001341
1  TEST_00001     0.002141
2  TEST_00002     0.153350
3  TEST_00003     0.107328
4  TEST_00004     0.509540


## 7. 실제 대회에 제출한 뒤 (점수 확인 후 실행)

**대회 사이트에 위에서 만든 csv를 업로드해서 실제 점수를 받은 뒤에만** 이 셀을 실행하세요.


In [ ]:
ACTUAL_SCORE = "0.0000"  # 대회에서 받은 실제 점수를 여기에 입력하세요

old_path = SAVE_PATH
new_name = f"{VERSION}차_{MODEL_NAME}_submit_improved({LOCAL_SCORE}->{ACTUAL_SCORE}).csv"
new_path = f"{SUBMISSION_DIR}/{new_name}"

os.rename(old_path, new_path)
print(f"파일명 변경 완료: {new_name}")

state = load_version_state()
state["submitted"] = True
save_version_state(state)
print(f"{VERSION}차 제출 완료로 표시됨. 다음 실행부터는 {VERSION + 1}차로 시작됩니다.")